In [ ]:
import math
import random
from decimal import Decimal, getcontext

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from shapely import affinity
from shapely.geometry import Polygon
from shapely.strtree import STRtree

# Global numeric settings

In [ ]:
pd.set_option("display.float_format", "{:.12f}".format)

# High precision for Decimal computations
getcontext().prec = 25

# Scale factor used to keep coordinates in integer-ish range for Shapely
scale_factor = Decimal("1e15")

# Build the index of the submission, in the format: "<trees_in_problem>_<tree_index>"
index = [f"{n:03d}_{t}" for n in range(1, 201) for t in range(n)]

# Packing configuration: ALL TUNING KNOBS LIVE HERE

In [ ]:
PACKING_CONFIG = {
    # Radial search towards the center for each new tree
    "start_radius": Decimal("20.0"),
    "step_in": Decimal("0.4"),
    # How many random rays/orientations we try per tree
    "attempts_per_tree": 300,
    # Collision behaviour
    "allow_touching": True,
}

# Tree geometry

In [ ]:
class ChristmasTree:
    """
    Represents a single, rotatable Christmas tree of a fixed size.

    Coordinates are stored at a high precision scale (scale_factor).
    Angle is in degrees.
    """

    def __init__(self, center_x="0", center_y="0", angle="0"):
        self.center_x = Decimal(center_x)
        self.center_y = Decimal(center_y)
        self.angle = Decimal(angle)  # degrees

        trunk_w = Decimal("0.15")
        trunk_h = Decimal("0.2")
        base_w = Decimal("0.7")
        mid_w = Decimal("0.4")
        top_w = Decimal("0.25")

        tip_y = Decimal("0.8")
        tier_1_y = Decimal("0.5")
        tier_2_y = Decimal("0.25")
        base_y = Decimal("0.0")
        trunk_bottom_y = -trunk_h

        # Define the tree polygon in local coordinates (scaled by scale_factor)
        initial_polygon = Polygon(
            [
                # Start at Tip
                (Decimal("0.0") * scale_factor, tip_y * scale_factor),
                # Right side - Top Tier
                (top_w / Decimal("2") * scale_factor, tier_1_y * scale_factor),
                (top_w / Decimal("4") * scale_factor, tier_1_y * scale_factor),
                # Right side - Middle Tier
                (mid_w / Decimal("2") * scale_factor, tier_2_y * scale_factor),
                (mid_w / Decimal("4") * scale_factor, tier_2_y * scale_factor),
                # Right side - Bottom Tier
                (base_w / Decimal("2") * scale_factor, base_y * scale_factor),
                # Right Trunk
                (trunk_w / Decimal("2") * scale_factor, base_y * scale_factor),
                (trunk_w / Decimal("2") * scale_factor, trunk_bottom_y * scale_factor),
                # Left Trunk
                (-(trunk_w / Decimal("2")) * scale_factor, trunk_bottom_y * scale_factor),
                (-(trunk_w / Decimal("2")) * scale_factor, base_y * scale_factor),
                # Left side - Bottom Tier
                (-(base_w / Decimal("2")) * scale_factor, base_y * scale_factor),
                # Left side - Middle Tier
                (-(mid_w / Decimal("4")) * scale_factor, tier_2_y * scale_factor),
                (-(mid_w / Decimal("2")) * scale_factor, tier_2_y * scale_factor),
                # Left side - Top Tier
                (-(top_w / Decimal("4")) * scale_factor, tier_1_y * scale_factor),
                (-(top_w / Decimal("2")) * scale_factor, tier_1_y * scale_factor),
            ]
        )

        # Store base (unrotated) polygon for easy re-rotation
        self.base_polygon = initial_polygon

        # Initialize actual polygon with current center and angle
        self.update_polygon()

    def update_polygon(self):
        """Recompute self.polygon given self.center_x, self.center_y, and self.angle."""
        rotated = affinity.rotate(self.base_polygon, float(self.angle), origin=(0, 0))
        self.polygon = affinity.translate(
            rotated,
            xoff=float(self.center_x * scale_factor),
            yoff=float(self.center_y * scale_factor),
        )

# Utility functions

In [ ]:
def generate_weighted_angle():
    """
    Generates a random angle in radians with distribution weighted by abs(sin(2*angle)).
    This helps place more trees in corners and makes the packing less "round".
    """
    while True:
        angle = random.uniform(0, 2 * math.pi)
        if random.uniform(0, 1) < abs(math.sin(2 * angle)):
            return angle


def build_candidate_polygon(tree: ChristmasTree, center_x: Decimal, center_y: Decimal, angle_deg: float):
    """
    Build a candidate Shapely polygon for a given tree at a hypothetical
    center and angle (degrees).
    """
    rotated = affinity.rotate(tree.base_polygon, angle_deg, origin=(0, 0))
    return affinity.translate(
        rotated,
        xoff=float(center_x * scale_factor),
        yoff=float(center_y * scale_factor),
    )


def collides(candidate_poly, placed_polygons, tree_index, cfg):
    """
    Collision check between candidate_poly and all already-placed polygons.
    Uses STRtree for speed.
    """
    if not placed_polygons:
        return False

    possible_indices = tree_index.query(candidate_poly)
    allow_touching = cfg["allow_touching"]

    for i in possible_indices:
        other = placed_polygons[i]
        if candidate_poly.intersects(other):
            if allow_touching:
                # Only count as collision if there's an actual overlap (not just boundary touch)
                if not candidate_poly.touches(other):
                    return True
            else:
                # Any intersection counts as collision
                return True

    return False


def compute_bounds_scaled(polygons):
    """
    Return (minx, miny, maxx, maxy) in *scaled* coordinates (Decimals)
    for a list of polygons.
    """
    if not polygons:
        return None

    first_bounds = polygons[0].bounds
    minx = Decimal(first_bounds[0])
    miny = Decimal(first_bounds[1])
    maxx = Decimal(first_bounds[2])
    maxy = Decimal(first_bounds[3])

    for poly in polygons[1:]:
        b = poly.bounds
        minx = min(minx, Decimal(b[0]))
        miny = min(miny, Decimal(b[1]))
        maxx = max(maxx, Decimal(b[2]))
        maxy = max(maxy, Decimal(b[3]))

    return minx, miny, maxx, maxy


def side_with_candidate(existing_bounds_scaled, cand_poly):
    """
    Given existing global bounds (scaled Decimals) and a candidate polygon,
    compute the resulting square side in *unscaled* units.
    """
    cb = cand_poly.bounds
    cminx = Decimal(cb[0])
    cminy = Decimal(cb[1])
    cmaxx = Decimal(cb[2])
    cmaxy = Decimal(cb[3])

    if existing_bounds_scaled is None:
        minx = cminx
        miny = cminy
        maxx = cmaxx
        maxy = cmaxy
    else:
        emin_x, emin_y, emax_x, emax_y = existing_bounds_scaled
        minx = min(emin_x, cminx)
        miny = min(emin_y, cminy)
        maxx = max(emax_x, cmaxx)
        maxy = max(emax_y, cmaxy)

    width_scaled = maxx - minx
    height_scaled = maxy - miny
    side_scaled = max(width_scaled, height_scaled)

    # convert back to tree space
    return side_scaled / scale_factor


def compute_bounding_side(polygons):
    """
    Convenience: compute bounding square side for plotting / reporting.
    """
    if not polygons:
        return Decimal("0")

    bounds = compute_bounds_scaled(polygons)
    minx, miny, maxx, maxy = bounds
    width = (maxx - minx) / scale_factor
    height = (maxy - miny) / scale_factor
    return max(width, height)


# Placement engine (greedy, square-aware)

In [ ]:
def initialize_trees(num_trees, existing_trees=None):
    """
    Build a greedy configuration by using the previous n-tree placement and adding
    one more tree to get the (n+1)-tree configuration.

    For each new tree:
      1) Fix all existing trees.
      2) Try many random rays & orientations.
      3) Along each ray, walk *inwards* toward the centre, checking collision.
      4) For every collision-free position, compute the resulting square side.
      5) Keep the candidate with the smallest square side.

    This directly optimizes the square size instead of radius.
    """
    cfg = PACKING_CONFIG

    if num_trees == 0:
        return [], Decimal("0")

    if existing_trees is None:
        placed_trees = []
    else:
        placed_trees = list(existing_trees)

    num_to_add = num_trees - len(placed_trees)

    if num_to_add > 0:
        # Create new trees with random initial orientations
        unplaced_trees = [ChristmasTree(angle=str(random.uniform(0, 360))) for _ in range(num_to_add)]

        # If starting from scratch, first tree at origin
        if not placed_trees:
            first_tree = unplaced_trees.pop(0)
            first_tree.center_x = Decimal("0")
            first_tree.center_y = Decimal("0")
            first_tree.update_polygon()
            placed_trees.append(first_tree)

        for tree_to_place in unplaced_trees:
            placed_polygons = [p.polygon for p in placed_trees]
            existing_bounds_scaled = compute_bounds_scaled(placed_polygons) if placed_polygons else None
            tree_index = STRtree(placed_polygons) if placed_polygons else None

            best_px = None
            best_py = None
            best_angle = None
            best_poly = None
            best_side = None

            for _ in range(cfg["attempts_per_tree"]):
                # Choose a random direction (ray) and a random tree orientation
                ray_angle = generate_weighted_angle()
                vx = Decimal(str(math.cos(ray_angle)))
                vy = Decimal(str(math.sin(ray_angle)))
                base_angle_deg = random.uniform(0, 360)

                radius = Decimal(cfg["start_radius"])
                step_in = cfg["step_in"]

                while radius >= 0:
                    px = radius * vx
                    py = radius * vy

                    cand_poly = build_candidate_polygon(tree_to_place, px, py, base_angle_deg)

                    if tree_index is not None and collides(cand_poly, placed_polygons, tree_index, cfg):
                        # hit something along this ray – stop moving inward
                        break

                    side = side_with_candidate(existing_bounds_scaled, cand_poly)

                    if (best_side is None) or (side < best_side):
                        best_side = side
                        best_px = px
                        best_py = py
                        best_angle = Decimal(str(base_angle_deg))
                        best_poly = cand_poly

                    radius -= step_in

            # Fallback if nothing found (should be extremely rare)
            if best_poly is None:
                best_px = Decimal("0")
                best_py = Decimal("0")
                best_angle = tree_to_place.angle
                best_poly = build_candidate_polygon(tree_to_place, best_px, best_py, float(best_angle))

            tree_to_place.center_x = best_px
            tree_to_place.center_y = best_py
            tree_to_place.angle = best_angle
            tree_to_place.polygon = best_poly

            placed_trees.append(tree_to_place)

    all_polygons = [t.polygon for t in placed_trees]
    side_length = compute_bounding_side(all_polygons)

    return placed_trees, side_length


# Plotting

In [ ]:
def plot_results(side_length, placed_trees, num_trees):
    """Plots the arrangement of trees and the bounding square."""
    fig, ax = plt.subplots(figsize=(6, 6))

    colors = plt.cm.viridis(np.linspace(0, 1, num_trees))

    all_polygons = [t.polygon for t in placed_trees]
    bounds_scaled = compute_bounds_scaled(all_polygons)
    minx_s, miny_s, maxx_s, maxy_s = bounds_scaled

    minx = minx_s / scale_factor
    miny = miny_s / scale_factor
    maxx = maxx_s / scale_factor
    maxy = maxy_s / scale_factor

    width = maxx - minx
    height = maxy - miny
    side_length = max(width, height)

    for i, tree in enumerate(placed_trees):
        x_scaled, y_scaled = tree.polygon.exterior.xy
        x = [Decimal(val) / scale_factor for val in x_scaled]
        y = [Decimal(val) / scale_factor for val in y_scaled]
        ax.plot(x, y, color=colors[i])
        ax.fill(x, y, alpha=0.5, color=colors[i])

    square_x = minx if width >= height else minx - (side_length - width) / 2
    square_y = miny if height >= width else miny - (side_length - height) / 2

    bounding_square = Rectangle(
        (float(square_x), float(square_y)),
        float(side_length),
        float(side_length),
        fill=False,
        edgecolor="red",
        linewidth=2,
        linestyle="--",
    )
    ax.add_patch(bounding_square)

    padding = Decimal("0.5")
    ax.set_xlim(
        float(square_x - padding),
        float(square_x + side_length + padding),
    )
    ax.set_ylim(
        float(square_y - padding),
        float(square_y + side_length + padding),
    )
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")
    plt.title(f"{num_trees} Trees: {side_length:.12f}")
    plt.show()
    plt.close(fig)


# Main script: build all configurations and write submission


In [ ]:
cfg = PACKING_CONFIG
tree_data = []
current_placed_trees = []  # carried forward so each n uses previous placement

for n in range(200):
    current_placed_trees, side = initialize_trees(n + 1, existing_trees=current_placed_trees)

    # Plot every 10 trees for sanity check
    if (n + 1) % 10 == 0:
        plot_results(side, current_placed_trees, n + 1)

    for tree in current_placed_trees:
        tree_data.append([tree.center_x, tree.center_y, tree.angle])

cols = ["x", "y", "deg"]
submission = pd.DataFrame(index=index, columns=cols, data=tree_data).rename_axis("id")

for col in cols:
    submission[col] = submission[col].astype(float).round(decimals=6)

for col in submission.columns:
    submission[col] = "s" + submission[col].astype("string")

# For Kaggle
submission.to_csv("/kaggle/working/submission.csv")
